# Task Mining — компактный сводный отчёт за все дни сотрудника

Использует прежний кеш анализа и консолидации. Финал: 3–5 предложений + единая таблица инсайтов, формируемая пакетами.


In [ ]:
%pip install openai httpx markdown tqdm

In [ ]:
from pathlib import Path
from datetime import datetime
import os, re, json, csv, hashlib, time, traceback
import httpx
from openai import OpenAI, RateLimitError, APIStatusError, APIConnectionError, APITimeoutError
from tqdm.auto import tqdm


In [ ]:
# Папка результатов ПРЕДЫДУЩЕГО ноутбука:
SOURCE_OUTPUT = Path('./output')
# Новая папка результатов по сотрудникам (не перезаписывает дневные отчёты):
MULTIDAY_OUTPUT = Path('./output_employees')

MODEL_NAME = 'CodegenChatReasoningUltra/codegen'
BASE_URL = 'https://ai-gateway.raisa.go.rshbank.ru/v1'
API_KEY = os.getenv('RAISA_API_KEY', '')  # или задайте ключ в переменной
VERIFY_SSL = True  # при необходимости корпоративной среды измените на False

MAX_MD_CHARS = 20_000              # размер части MD для первичного анализа
MAX_MERGE_CHARS = 35_000           # размер блока промежуточной консолидации
PART_MAX_TOKENS = 3_000
MERGE_MAX_TOKENS = 4_000
FINAL_MAX_TOKENS = 12_000  # прежнее значение; секции используют SECTION_MAX_TOKENS
INTRO_MAX_TOKENS = 2000
INSIGHT_EXTRACT_MAX_TOKENS = 3500
INSIGHT_BATCH_MAX_TOKENS = 4000
INSIGHT_SOURCE_CHARS = 9500
INSIGHTS_PER_BATCH = 5
REQUEST_TIMEOUT_SECONDS = 300  # клиентский тайм-аут; серверный 504 может наступить раньше
MAX_RETRIES = 3
REQUEST_DELAY_SECONDS = 20
RATE_LIMIT_WAIT_SECONDS = 65
ENABLE_THINKING_INTERMEDIATE = False
ENABLE_THINKING_FINAL = False
SHOW_PROGRESS = True

# ВСТАВЬТЕ СВОЙ ПРОМПТ: он применяется на ФИНАЛЬНОМ этапе для каждого сотрудника.
MULTIDAY_ANALYSIS_PROMPT = """
"""

# Промпты технических этапов можно при необходимости отредактировать.
PART_PROMPT = """Ты анализируешь ФРАГМЕНТ уже готового Markdown-отчёта Task Mining за один рабочий день.
Извлеки все существенные выводы, подтверждения, числовые показатели, примеры и ограничения.
Сохраняй источник (имя файла и диапазон фрагмента). Не придумывай данные.
Пиши компактно, в Markdown. Не формируй финальный отчёт за все дни.
Если фрагмент содержит лишь часть таблицы, не додумывай отсутствующие колонки."""
MERGE_PROMPT = """Ты консолидируешь аналитические наблюдения Task Mining за разные дни одного сотрудника.
Объединяй повторяющиеся проблемы и рекомендации, сохраняя уникальные наблюдения, даты/источники,
числовые значения и ограничения. Не суммируй пересекающиеся показатели и не считай упоминания
частотой события. Отделяй устойчивые закономерности от разовых наблюдений.
Не формируй окончательный отчёт. Верни компактный Markdown без потери существенных фактов."""

MULTIDAY_OUTPUT.mkdir(parents=True, exist_ok=True)

# Итог: короткое описание + одна таблица инсайтов, без ограничения числа строк.
# Исходный пользовательский промпт применяется на финальных этапах.


In [ ]:
if not MULTIDAY_ANALYSIS_PROMPT.strip():
    raise ValueError('Заполните MULTIDAY_ANALYSIS_PROMPT в ячейке настроек.')
if not API_KEY:
    raise ValueError('Укажите RAISA_API_KEY (или заполните API_KEY в настройках).')
llm_client = OpenAI(base_url=BASE_URL, api_key=API_KEY,
                    http_client=httpx.Client(verify=VERIFY_SSL, timeout=REQUEST_TIMEOUT_SECONDS),
                    max_retries=0)
print('Клиент создан. Проверка списка моделей не обязательна.')


In [ ]:
def render_html(markdown_text: str) -> str:
    """Преобразует Markdown модели в автономный HTML с таблицами и стилями."""
    import markdown as md

    # Удаляем внешнюю обёртку ```markdown, если модель вернула её целиком.
    clean = markdown_text.strip().replace("\r\n", "\n")
    match = re.fullmatch(r"```(?:markdown|md)?\s*\n(.*?)\n```", clean, flags=re.I | re.S)
    if match:
        clean = match.group(1)

    # Markdown требует пустую строку перед таблицей и после неё.
    lines = clean.splitlines()
    fixed = []
    for i, line in enumerate(lines):
        is_table_start = (
            "|" in line and i + 1 < len(lines)
            and bool(re.match(r"^\s*\|?\s*:?-{3,}:?", lines[i + 1]))
        )
        if is_table_start and fixed and fixed[-1].strip():
            fixed.append("")
        fixed.append(line)
    clean = "\n".join(fixed)
    body = md.markdown(clean, extensions=["tables", "fenced_code", "sane_lists", "nl2br"])

    return """<!doctype html>
<html lang="ru"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Анализ Task Mining</title>
<style>
*{box-sizing:border-box}
body{margin:0;padding:32px;background:#f0f5f1;color:#24382b;font:15px/1.65 Arial,Helvetica,sans-serif}
main{max-width:1450px;margin:0 auto;padding:38px 44px;background:#fff;border-radius:14px;box-shadow:0 8px 30px #173d2515}
h1,h2,h3,h4{color:#17623b;line-height:1.3;margin:1.5em 0 .65em}
h1{font-size:27px;border-bottom:3px solid #3d9d64;padding-bottom:14px;margin-top:0}
h2{font-size:21px;border-bottom:1px solid #dce9df;padding-bottom:8px}
h3{font-size:17px}
p{margin:0 0 14px}ul,ol{padding-left:26px}li{margin:5px 0}
.table-wrap{width:100%;overflow-x:auto;margin:22px 0;border:1px solid #d5e4d9;border-radius:9px}
table{border-collapse:collapse;width:100%;min-width:900px;table-layout:auto;font-size:13px;line-height:1.5}
th,td{border:1px solid #dce8df;padding:12px 13px;text-align:left;vertical-align:top;overflow-wrap:anywhere;white-space:normal;min-width:140px}
th{background:#246e43;color:#fff;font-weight:700;min-width:160px}
tr:nth-child(even) td{background:#f4f9f5}tr:hover td{background:#eaf5ed}
pre{white-space:pre-wrap;overflow-wrap:anywhere;background:#f2f7f3;padding:16px;border-radius:8px}
code{overflow-wrap:anywhere}blockquote{border-left:4px solid #419c64;padding:4px 16px;margin:15px 0;background:#f5faf6}
@media(max-width:700px){body{padding:8px}main{padding:20px 15px}h1{font-size:22px}}
@media print{body{background:white;padding:0}main{box-shadow:none;padding:0;max-width:none}.table-wrap{overflow:visible}table{min-width:0;font-size:10px}th,td{min-width:0;padding:6px}thead{display:table-header-group}tr{break-inside:avoid}h1,h2,h3{break-after:avoid}}
</style></head><body><main>""" + re.sub(
        r"(<table>.*?</table>)", r'<div class="table-wrap">\1</div>', body, flags=re.S
    ) + "</main></body></html>"


In [ ]:
# Вспомогательные функции, кеш и журнал производительности
LOG_FIELDS = ['timestamp','employee','stage','part','attempt','status','seconds',
              'prompt_tokens','completion_tokens','reasoning_tokens','finish_reason','wait_seconds','error']

def perf(**kw):
    path = EMPLOYEE_DIR / 'performance_log.csv'
    path.parent.mkdir(parents=True, exist_ok=True)
    fresh = not path.exists() or path.stat().st_size == 0
    with path.open('a', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=LOG_FIELDS)
        if fresh: writer.writeheader()
        writer.writerow({k:kw.get(k,'') for k in LOG_FIELDS})

def cache_request(stage, part, system, user, max_tokens):
    thinking = ENABLE_THINKING_FINAL if stage.startswith('final') else ENABLE_THINKING_INTERMEDIATE
    payload = json.dumps({'model':MODEL_NAME,'stage':stage,'system':system,'user':user,
                          'max_tokens':max_tokens,'thinking':thinking}, ensure_ascii=False, sort_keys=True)
    key = hashlib.sha256(payload.encode('utf-8')).hexdigest()
    cache = EMPLOYEE_DIR / 'tm_cache' / (key + '.md')
    if cache.exists():
        result = cache.read_text(encoding='utf-8')
        if result.strip():
            print(f'  [КЕШ] {stage} {part}')
            return result
    for attempt in range(1, MAX_RETRIES + 1):
        print(f'  [НАЧАЛО] {stage} {part}, попытка {attempt}/{MAX_RETRIES}, max_tokens={max_tokens}', flush=True)
        start = time.perf_counter()
        try:
            response = llm_client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{'role':'system','content':system},{'role':'user','content':user}],
                max_tokens=max_tokens,
                extra_body={'chat_template_kwargs': {'enable_thinking':thinking}},
            )
            seconds = round(time.perf_counter()-start,2)
            choice = response.choices[0]
            finish = choice.finish_reason
            result = (choice.message.content or '').strip()
            usage = response.usage
            details = getattr(usage,'completion_tokens_details',None)
            status = 'truncated' if finish == 'length' else ('success' if result else 'empty')
            perf(employee=EMPLOYEE_NAME,stage=stage,part=part,attempt=attempt,status=status,
                 seconds=seconds,prompt_tokens=getattr(usage,'prompt_tokens',None),
                 completion_tokens=getattr(usage,'completion_tokens',None),
                 reasoning_tokens=getattr(details,'reasoning_tokens',None),finish_reason=finish)
            print(f'  [{stage} {part}] {seconds}с; input={getattr(usage,"prompt_tokens",None)} '
                  f'output={getattr(usage,"completion_tokens",None)}; finish={finish}')
            if finish == 'length':
                raise RuntimeError(f'ОТВЕТ ОБРЕЗАН: {stage} {part}; max_tokens={max_tokens}. '
                                   'Увеличьте лимит или уменьшите размер блоков.')
            if not result:
                raise RuntimeError(f'Пустой ответ: {stage} {part}')
            cache.parent.mkdir(parents=True,exist_ok=True)
            cache.write_text(result,encoding='utf-8')
            return result
        except RateLimitError as exc:
            wait = RATE_LIMIT_WAIT_SECONDS if attempt < MAX_RETRIES else 0
            perf(employee=EMPLOYEE_NAME,stage=stage,part=part,attempt=attempt,
                 status='http_429',seconds=round(time.perf_counter()-start,2),wait_seconds=wait,error=str(exc)[:1000])
        except (APIConnectionError, APITimeoutError) as exc:
            wait = min(10*attempt,60) if attempt < MAX_RETRIES else 0
            perf(employee=EMPLOYEE_NAME,stage=stage,part=part,attempt=attempt,
                 status=type(exc).__name__,seconds=round(time.perf_counter()-start,2),wait_seconds=wait,error=str(exc)[:1000])
        except APIStatusError as exc:
            wait = min(10*attempt,60) if exc.status_code >= 500 and attempt < MAX_RETRIES else 0
            perf(employee=EMPLOYEE_NAME,stage=stage,part=part,attempt=attempt,
                 status=f'http_{exc.status_code}',seconds=round(time.perf_counter()-start,2),wait_seconds=wait,error=str(exc)[:1500])
            if exc.status_code < 500: raise
        if not wait: raise RuntimeError(f'Ошибка API после попытки {attempt}: {stage} {part}; подробности в performance_log.csv')
        print(f'  [ОШИБКА API] {stage} {part}, попытка {attempt}; ожидание {wait} с. Подробности: performance_log.csv', flush=True)
        time.sleep(wait)
    raise RuntimeError(f'Превышено число повторов: {stage} {part}')

def split_md(text, max_chars):
    # По абзацам; длинные абзацы режутся по строкам, при необходимости по символам.
    blocks = re.split(r'(?<=\n)\n+',text.strip())
    units=[]
    for block in blocks:
        if len(block)<=max_chars:
            units.append(block);continue
        for line in block.splitlines(keepends=True):
            if len(line)>max_chars:
                units.extend(line[i:i+max_chars] for i in range(0,len(line),max_chars))
            else: units.append(line)
    chunks=[];current=''
    for unit in units:
        separator='\n\n' if current else ''
        if current and len(current)+len(separator)+len(unit)>max_chars:
            chunks.append(current);current='';separator=''
        current+=separator+unit
    if current: chunks.append(current)
    return chunks

def pause():
    if REQUEST_DELAY_SECONDS>0:
        perf(employee=EMPLOYEE_NAME,stage='pause',status='fixed_delay',wait_seconds=REQUEST_DELAY_SECONDS)
        time.sleep(REQUEST_DELAY_SECONDS)


In [ ]:
# Источники: output/<сотрудник>/<имя_активности>/<имя_активности>_analysis.md
# Вложенные папки внутри сотрудника также поддерживаются.
if not SOURCE_OUTPUT.is_dir():
    raise FileNotFoundError(f'Нет папки предыдущих результатов: {SOURCE_OUTPUT.resolve()}')

def find_reports():
    result={}
    for md_file in SOURCE_OUTPUT.rglob('*_analysis.md'):
        relative=md_file.relative_to(SOURCE_OUTPUT)
        if len(relative.parts)<3: continue
        employee=relative.parts[0]
        # Игнорируем случайно помещённые в output сводные отчёты.
        if md_file.stem.endswith('_all_days_analysis'): continue
        result.setdefault(employee,[]).append(md_file)
    return {k:sorted(v,key=lambda p:p.as_posix().casefold()) for k,v in sorted(result.items())}

EMPLOYEE_REPORTS = find_reports()
print('Сотрудников:',len(EMPLOYEE_REPORTS))
for employee,paths in EMPLOYEE_REPORTS.items():
    print(f'  {employee}: {len(paths)} дневных MD')
if not EMPLOYEE_REPORTS:
    raise FileNotFoundError('Не найдены дневные *_analysis.md в output/<сотрудник>/...')


In [ ]:
# ПОЛНЫЙ ЦИКЛ: сотрудник -> все дни -> консолидация -> один итоговый MD + HTML
# Обработка последовательная. Дневные отчёты в SOURCE_OUTPUT не изменяются.
success=[];failures=[]
for employee,paths in tqdm(EMPLOYEE_REPORTS.items(),desc='Сотрудники',unit='сотр.'):
    EMPLOYEE_NAME=employee
    EMPLOYEE_DIR=MULTIDAY_OUTPUT/employee
    EMPLOYEE_DIR.mkdir(parents=True,exist_ok=True)
    print(f'\n=== {employee}: {len(paths)} дневных отчётов ===')
    try:
        observations=[]
        for day_idx,path in enumerate(tqdm(paths,desc=f'Дни {employee}',unit='день'),1):
            source=path.relative_to(SOURCE_OUTPUT).as_posix()
            report=path.read_text(encoding='utf-8-sig').strip()
            if not report: raise ValueError(f'Пустой отчёт: {source}')
            chunks=split_md(report,MAX_MD_CHARS)
            for part_idx,chunk in enumerate(chunks,1):
                user=f'Сотрудник: {employee}\nИсточник: {source}\nФрагмент: {part_idx}/{len(chunks)}\n\n{chunk}'
                note=cache_request('part',f'{day_idx}.{part_idx}',PART_PROMPT,user,PART_MAX_TOKENS)
                observations.append(f'## Источник {source} / фрагмент {part_idx}\n\n{note}')
                pause()
        material='\n\n'.join(observations)
        level=0
        while len(material)>MAX_MERGE_CHARS:
            level+=1
            if level>20: raise RuntimeError('Слишком много уровней консолидации')
            blocks=split_md(material,MAX_MERGE_CHARS)
            print(f'Консолидация {employee}, уровень {level}: {len(blocks)} блоков')
            merged=[]
            for i,block in enumerate(tqdm(blocks,desc=f'Merge {level}',unit='блок'),1):
                user=f'Сотрудник: {employee}\nУровень {level}, блок {i}/{len(blocks)}\n\n{block}'
                merged.append(cache_request('merge',f'{level}.{i}',MERGE_PROMPT,user,MERGE_MAX_TOKENS))
                pause()
            new_material='\n\n'.join(f'## Консолидация {level}.{i}\n\n{x}' for i,x in enumerate(merged,1))
            if len(new_material)>=len(material) and len(new_material)>MAX_MERGE_CHARS:
                raise RuntimeError('Консолидация не уменьшает объём материала')
            material=new_material
        (EMPLOYEE_DIR/'consolidated_material.md').write_text(material,encoding='utf-8')
        source_list='\n'.join(f'- {p.relative_to(SOURCE_OUTPUT).as_posix()}' for p in paths)
        final_system=('Ты эксперт Task Mining. Возвращай только запрошенный результат в Markdown. '
                      'Не раскрывай рассуждения, не выдумывай факты, числа или источники. '
                      'Не повторяй входные данные без необходимости.')
        common=(f'Сотрудник: {employee}\nРабочих дней: {len(paths)}\n'
                f'Источники:\n{source_list}\n\n'
                f'Методология пользователя:\n{MULTIDAY_ANALYSIS_PROMPT}\n\n')

        # 1. Компактное описание деятельности — не более пяти предложений.
        print('\n[ФИНАЛ] Краткая характеристика', flush=True)
        intro_user=(common + 'ЗАДАЧА: напиши ТОЛЬКО 3–5 предложений о деятельности сотрудника '
                    'за весь период. Без таблиц, инсайтов, рекомендаций и расчётов. '
                    'Не пытайся выполнить другие пункты методологии в этом ответе.\n\n'
                    'МАТЕРИАЛ:\n'+material)
        intro=cache_request('final_intro_compact','1',final_system,intro_user,INTRO_MAX_TOKENS)
        pause()

        # 2. Извлечение уникальных кандидатов на инсайты из частей материала.
        # Не задаём максимальное количество инсайтов; каждый кандидат получает ID.
        # Если источник слишком велик, каждый фрагмент анализируется отдельно.
        chunks=split_md(material,INSIGHT_SOURCE_CHARS)
        print(f'[ФИНАЛ] Извлечение инсайтов: {len(chunks)} фрагментов',flush=True)
        candidates=[]
        for i,chunk in enumerate(tqdm(chunks,desc='Инсайты',unit='фрагмент'),1):
            extract_user=(common + f'ФРАГМЕНТ {i}/{len(chunks)}.\n'
                'Извлеки ВСЕ самостоятельные проблемы/неоптимальные действия из этого фрагмента. '
                'Ответ: нумерованный список. Для КАЖДОГО пункта укажи: '
                'краткое название; конкретные действия и доказательства; цифры и их единицы; '
                'источники; существующие рекомендации; ограничения расчёта эффекта. '
                'Не включай описание деятельности, не делай таблицу и не повторяй похожие пункты '
                'внутри этого фрагмента. Не выдумывай сведения. Пиши максимально компактно.\n\n'
                'ИСХОДНЫЙ ФРАГМЕНТ:\n'+chunk)
            extracted=cache_request('final_insight_extract_v2',str(i),final_system,
                                    extract_user,INSIGHT_EXTRACT_MAX_TOKENS)
            candidates.append(f'### Фрагмент {i}\n\n{extracted}')
            if i<len(chunks): pause()
        raw_candidates='\n\n'.join(candidates)
        (EMPLOYEE_DIR/'insights_extracted.md').write_text(raw_candidates,encoding='utf-8')

        # 3. Сведение повторов небольшими блоками с сохранением уникальных фактов.
        # На этом этапе не создаём финальную таблицу, чтобы не тратить токены на её оформление.
        dedup_material=raw_candidates
        dedup_level=0
        while len(dedup_material)>14000:
            dedup_level+=1
            if dedup_level>12: raise RuntimeError('Не удалось сократить список инсайтов')
            parts=split_md(dedup_material,11000)
            compact=[]
            print(f'[ФИНАЛ] Устранение повторов: уровень {dedup_level}, блоков {len(parts)}')
            for i,part in enumerate(parts,1):
                prompt=('Сведи совпадающие по смыслу инсайты в один, сохраняя все уникальные '
                        'доказательства, источники, количественные показатели и ограничения. '
                        'Не удаляй уникальные инсайты ради краткости. '
                        'Ответ — нумерованный список компактных пунктов, не таблица.\n\n'+part)
                compact.append(cache_request('final_dedup_v2',f'{dedup_level}.{i}',
                                             final_system,prompt,3500))
                if i<len(parts):pause()
            updated='\n\n'.join(compact)
            if len(updated)>=len(dedup_material):
                # Не допускаем бесконечную консолидацию и молчаливую потерю пунктов.
                break
            dedup_material=updated
        (EMPLOYEE_DIR/'insights_consolidated.md').write_text(dedup_material,encoding='utf-8')

        # 4. Группируем инсайты в пакеты по отдельным пунктам. Если единый
        # список не удалось компактно свести, используем отдельные блоки.
        # Это не ограничивает количество инсайтов.
        def split_insight_items(text):
            lines=text.splitlines()
            items=[]; current=[]
            pattern=re.compile(r'^\s*(?:\d+[.)]|[-*]\s+|#{2,4}\s+)',re.UNICODE)
            for line in lines:
                if pattern.match(line) and current:
                    items.append('\n'.join(current).strip());current=[line]
                else: current.append(line)
            if current:items.append('\n'.join(current).strip())
            return [x for x in items if x and not re.fullmatch(r'#{1,6}.*',x.strip())]

        items=split_insight_items(dedup_material)
        if not items: raise RuntimeError('Не удалось извлечь инсайты из консолидированного материала')
        batches=[]; batch=[]
        for item in items:
            if batch and (len(batch)>=INSIGHTS_PER_BATCH or
                          len('\n'.join(batch))+len(item)>10500):
                batches.append(batch);batch=[]
            batch.append(item)
        if batch:batches.append(batch)
        print(f'[ФИНАЛ] Формирование таблицы: {len(items)} кандидатов, {len(batches)} пакетов')
        header=('| Выявленная неоптимальная цепочка действий / действие | Описание проблемы | '
                'Предложение быстрого устранения | Стратегическое воздействие | '
                'Оценка потенциального эффекта |\n'
                '|---|---|---|---|---|')
        # Нормализация ответов: модель иногда добавляет номер строки перед пятью колонками,
        # не ставит крайние | или возвращает Markdown-таблицу с заголовком.
        def parse_table_rows(answer):
            found=[]
            for raw in answer.splitlines():
                line=raw.strip()
                if not line or '|' not in line: continue
                line=re.sub(r'^\s*(?:\d+[.)]\s*)?(?=\|)', '', line)
                cells=[c.strip() for c in line.strip('|').split('|')]
                if len(cells)==6 and re.fullmatch(r'(?:№\s*)?\d+[.)]?',cells[0],re.I):
                    cells=cells[1:]
                if len(cells)==5 and not all(re.fullmatch(r':?-{3,}:?',c or '') for c in cells):
                    if 'описание проблемы' in line.lower() and 'стратегическ' in line.lower():
                        continue
                    found.append('| '+' | '.join(c.replace('\n',' ').strip() for c in cells)+' |')
            return found

        rows=[]
        section_dir=EMPLOYEE_DIR/'final_sections'
        section_dir.mkdir(parents=True,exist_ok=True)
        incomplete=[]
        for i,batch in enumerate(batches,1):
            print(f'[ФИНАЛ] Таблица: пакет {i}/{len(batches)}',flush=True)
            table_user=(common +
                'Сформируй ТОЛЬКО строки Markdown-таблицы (БЕЗ заголовка и разделителя). '
                'Пять колонок строго в таком порядке: 1) неоптимальная цепочка/действие '
                'с примерами; 2) проблема; 3) быстрое устранение; 4) стратегическое '
                'воздействие; 5) потенциальный эффект и формула при наличии данных. '
                'Каждый самостоятельный уникальный инсайт — отдельная строка. '
                'Не пропускай инсайты, не добавляй вступления или выводы. '
                'Не придумывай числа; если данных для оценки нет, укажи, что расчёт '
                'невозможен, и какие показатели нужны. Не ставь символ | внутри ячейки. '
                'Пиши компактно, но сохраняй конкретные подтверждения. '
                'Не пытайся составить весь отчёт: это только один пакет.\n\n'
                'ИНСАЙТЫ ПАКЕТА:\n'+'\n\n'.join(batch))
            table=cache_request('final_table_batch_v2',str(i),final_system,
                                table_user,INSIGHT_BATCH_MAX_TOKENS)
            clean=parse_table_rows(table)
            if not clean:
                # Ответ сохранён отдельно; исправляем только его формат без повторного анализа.
                (section_dir/f'table_batch_{i}_raw.md').write_text(table,encoding='utf-8')
                repair_system='Ты редактор Markdown-таблиц. Верни только строки с ровно пятью колонками.'
                repair_user=('Переформатируй следующий ответ в строки Markdown-таблицы с 5 колонками: '
                             'действие | проблема | быстрое устранение | стратегическое воздействие | эффект. '
                             'Сохрани все факты и цифры. Без заголовка, нумерации и пояснений. '
                             'Если это не таблица, восстанови строки по тексту.\n\n'+table)
                try:
                    repaired=cache_request('final_table_repair_v1',str(i),repair_system,
                                           repair_user,INSIGHT_BATCH_MAX_TOKENS)
                    clean=parse_table_rows(repaired)
                except Exception as repair_exc:
                    print(f'⚠ Пакет {i}: ошибка исправления: {repair_exc}',flush=True)
            if not clean:
                incomplete.append(i)
                print(f'⚠ Пакет {i}: некорректная таблица, сохранён исходный ответ; продолжаем',flush=True)
            else:
                rows.extend(clean)
                (section_dir/f'table_batch_{i}.md').write_text('\n'.join(clean),encoding='utf-8')
            if i<len(batches):pause()
        if incomplete:
            (EMPLOYEE_DIR/'incomplete_batches.txt').write_text(
                'Не удалось разобрать пакеты: '+', '.join(map(str,incomplete))+'\n'
                'Проверьте *_raw.md в final_sections. Отчёт НЕПОЛНЫЙ.\n',encoding='utf-8')
        else:
            (EMPLOYEE_DIR/'incomplete_batches.txt').unlink(missing_ok=True)
        final=(f'# Консолидированный анализ активности: {employee}\n\n'
               f'Рабочих дней: {len(paths)}\n\n'
               f'## 1. Краткая характеристика деятельности\n\n{intro.strip()}\n\n'
               f'## 2. Выявленные возможности оптимизации\n\n{header}\n'
               +'\n'.join(rows)+'\n')
        if incomplete:
            final=('> ⚠ **НЕПОЛНЫЙ ОТЧЁТ:** не обработаны пакеты '+', '.join(map(str,incomplete))+'; см. incomplete_batches.txt.\n\n')+final
        md_path=EMPLOYEE_DIR/f'{employee}_all_days_analysis.md'
        html_path=EMPLOYEE_DIR/f'{employee}_all_days_analysis.html'
        md_path.write_text(final,encoding='utf-8')
        html_path.write_text(render_html(final),encoding='utf-8')
        if incomplete:
            failures.append((employee,'Неполная таблица: пакеты '+', '.join(map(str,incomplete))))
        else:
            success.append(employee)
        print('ГОТОВО:',html_path.resolve())
    except Exception as exc:
        (EMPLOYEE_DIR/'error.txt').write_text(traceback.format_exc(),encoding='utf-8')
        failures.append((employee,str(exc)))
        print(f'ОШИБКА {employee}: {type(exc).__name__}: {exc}')
print(f'\nИТОГО: успешно {len(success)}, ошибок {len(failures)}')
for name,error in failures:print(' -',name,error)


In [ ]:
# Только обновление HTML из уже готовых итоговых MD — без вызовов LLM.
for md_path in MULTIDAY_OUTPUT.rglob('*_all_days_analysis.md'):
    html_path=md_path.with_suffix('.html')
    html_path.write_text(render_html(md_path.read_text(encoding='utf-8')),encoding='utf-8')
    print('HTML обновлён:',html_path)
